# Trial simulations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from statsmodels.tsa.regime_switching.markov_autoregression import MarkovAutoregression
import warnings

# Suppress convergence warnings during the loop
warnings.filterwarnings("ignore")

np.random.seed(42)

In [ ]:
num_trials = 10
N = 10_000
burn = 500

# Tracking Information Criteria for all 3 models
ic_records = {
    'AR(1)':       {'aic': [], 'bic': []},
    'MS-Reg':      {'aic': [], 'bic': []},
    'MS-AutoReg':  {'aic': [], 'bic': []}
}

# Tracking parameter recovery (just for the final MS-AutoReg model)
results = {
    'phi_0': {'true': [], 'est': []}, 'phi_1': {'true': [], 'est': []},
    'sig_0': {'true': [], 'est': []}, 'sig_1': {'true': [], 'est': []},
    'p00':   {'true': [], 'est': []}, 'p11':   {'true': [], 'est': []}
}

In [16]:
print(f"Running {num_trials} trials comparing AR(1), MS-Regression, and MS-Autoregression...")

for trial in range(num_trials):
    print(f"  -> Fitting Trial {trial + 1}/{num_trials}...")
    
    # 1. Generate Random True Parameters
    p00_true = np.random.uniform(0.95, 0.99)
    p11_true = np.random.uniform(0.85, 0.92)
    P = np.array([[p00_true, 1 - p00_true], 
                  [1 - p11_true, p11_true]])
    
    mu_true = np.array([0.0, 0.0])
    phi_true = np.array([np.random.uniform(0.1, 0.4), np.random.uniform(0.7, 0.9)])
    sig_true = np.array([np.random.uniform(0.3, 0.6), np.random.uniform(1.5, 2.5)])
    
    # 2. Simulate Data
    states = np.zeros(N, dtype=int)
    y = np.zeros(N)
    for t in range(1, N):
        states[t] = np.random.choice([0, 1], p=P[states[t-1]])
        s = states[t]
        y[t] = mu_true[s] + phi_true[s] * y[t-1] + sig_true[s] * np.random.randn()
        
    y_train = y[burn:]
    
    # --- MODEL 1: Baseline AR(1) ---
    ar_model = AutoReg(y_train, lags=1, trend="c")
    ar_res = ar_model.fit()
    ic_records['AR(1)']['aic'].append(ar_res.aic)
    ic_records['AR(1)']['bic'].append(ar_res.bic)
    
    # Prepare manual lags for MS-Regression
    y_dep = y_train[1:]
    y_lag = y_train[:-1].reshape(-1, 1)
    
    # --- MODEL 2: MarkovRegression (Manual Lags) ---
    ms_reg_model = MarkovRegression(
        endog=y_dep, k_regimes=2, trend="c", exog=y_lag,
        switching_trend=True, switching_exog=True, switching_variance=True
    )
    # Using lower search reps for speed in the loop
    ms_reg_res = ms_reg_model.fit(search_reps=10, em_iter=15, disp=False)
    ic_records['MS-Reg']['aic'].append(ms_reg_res.aic)
    ic_records['MS-Reg']['bic'].append(ms_reg_res.bic)

    # --- MODEL 3: MarkovAutoregression (Built-in Lags) ---
    ms_ar_model = MarkovAutoregression(
        endog=y_train, k_regimes=2, order=1, trend="c",
        switching_ar=True, switching_trend=True, switching_variance=True
    )
    ms_ar_res = ms_ar_model.fit(search_reps=15, em_iter=20, disp=False)
    ic_records['MS-AutoReg']['aic'].append(ms_ar_res.aic)
    ic_records['MS-AutoReg']['bic'].append(ms_ar_res.bic)
    
    # 3. Retrieve Parameters for MS-AutoReg (Handle Label Switching)
    params_dict = dict(zip(ms_ar_res.model.param_names, ms_ar_res.params))
    
    est_var_0 = params_dict.get('sigma2[0]', np.nan)
    est_var_1 = params_dict.get('sigma2[1]', np.nan)
    map_0, map_1 = (0, 1) if est_var_0 < est_var_1 else (1, 0)
        
    results['phi_0']['true'].append(phi_true[0])
    results['phi_0']['est'].append(params_dict.get(f'ar.L1[{map_0}]', np.nan))
    results['phi_1']['true'].append(phi_true[1])
    results['phi_1']['est'].append(params_dict.get(f'ar.L1[{map_1}]', np.nan))
    results['sig_0']['true'].append(sig_true[0])
    results['sig_0']['est'].append(np.sqrt(params_dict.get(f'sigma2[{map_0}]', np.nan)))
    results['sig_1']['true'].append(sig_true[1])
    results['sig_1']['est'].append(np.sqrt(params_dict.get(f'sigma2[{map_1}]', np.nan)))
    results['p00']['true'].append(p00_true)
    results['p11']['true'].append(p11_true)
    if map_0 == 0:
        results['p00']['est'].append(params_dict.get('p[0->0]', np.nan))
        results['p11']['est'].append(1.0 - params_dict.get('p[1->0]', np.nan))
    else:
        results['p00']['est'].append(1.0 - params_dict.get('p[1->0]', np.nan))
        results['p11']['est'].append(params_dict.get('p[0->0]', np.nan))

print("\n" + "="*60)
print(" MODEL COMPARISON (AVERAGE OVER 10 TRIALS) ")
print("="*60)

KeyboardInterrupt: 

In [ ]:
# ------------------------------------------------------------
# 4. Print AIC/BIC Summaries
# ------------------------------------------------------------
comp_data = []
for model_name in ic_records.keys():
    avg_aic = np.mean(ic_records[model_name]['aic'])
    avg_bic = np.mean(ic_records[model_name]['bic'])
    comp_data.append({'Model': model_name, 'Avg_AIC': avg_aic, 'Avg_BIC': avg_bic})

comp_df = pd.DataFrame(comp_data)
print(comp_df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
print("\n* Note: Lower AIC/BIC indicates a better model fit.\n")

In [ ]:
# ------------------------------------------------------------
# 5. Plotting AIC/BIC Comparison & Parameter Recovery
# ------------------------------------------------------------
fig = plt.figure(figsize=(14, 12))

# --- Plot 1: Model Comparison (Bar Chart) ---
ax_bar = plt.subplot2grid((3, 2), (0, 0), colspan=2)
x = np.arange(3)
width = 0.35
avg_aics = comp_df['Avg_AIC'].values
avg_bics = comp_df['Avg_BIC'].values

ax_bar.bar(x - width/2, avg_aics, width, label='Average AIC', color='skyblue', edgecolor='k')
ax_bar.bar(x + width/2, avg_bics, width, label='Average BIC', color='salmon', edgecolor='k')

ax_bar.set_ylabel('Information Criteria Value')
ax_bar.set_title('Model Performance Comparison (Lower is Better)')
ax_bar.set_xticks(x)
ax_bar.set_xticklabels(comp_df['Model'])
ax_bar.legend()
ax_bar.grid(axis='y', alpha=0.3)

# --- Plot 2: Parameter Recovery Scatter Plots ---
metrics = [
    ('phi_0', 'AR Coefficient (Calm)'),
    ('phi_1', 'AR Coefficient (Turbulent)'),
    ('sig_0', 'Volatility (Calm)'),
    ('sig_1', 'Volatility (Turbulent)')
]

for i, (key, title) in enumerate(metrics):
    # Adjust subplot indexing so it sits below the bar chart
    ax = plt.subplot2grid((3, 2), (1 + i//2, i%2))
    
    true_vals = results[key]['true']
    est_vals = results[key]['est']
    ax.scatter(true_vals, est_vals, color='blue', alpha=0.7, edgecolor='k', s=50)
    
    lims = [
        np.min([ax.get_xlim(), ax.get_ylim()]),  
        np.max([ax.get_xlim(), ax.get_ylim()])
    ]
    ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0)
    ax.set_title(title)
    ax.set_xlabel('True Parameter Value')
    ax.set_ylabel('Estimated Parameter Value')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# 6. Detailed Parameter Printouts
# ------------------------------------------------------------

print("\n" + "="*90)
print(" PARAMETER RECOVERY: TRUE vs ESTIMATED (Trial-by-Trial) ")
print("="*90)

# Build a comprehensive DataFrame tracking every single parameter
df_params = pd.DataFrame({
    'Trial': np.arange(1, num_trials + 1),
    
    'True_AR_Calm': results['phi_0']['true'],
    'Est_AR_Calm':  results['phi_0']['est'],
    'True_AR_Turb': results['phi_1']['true'],
    'Est_AR_Turb':  results['phi_1']['est'],
    
    'True_Vol_Calm': results['sig_0']['true'],
    'Est_Vol_Calm':  results['sig_0']['est'],
    'True_Vol_Turb': results['sig_1']['true'],
    'Est_Vol_Turb':  results['sig_1']['est'],
    
    'True_p00': results['p00']['true'],
    'Est_p00':  results['p00']['est'],
    'True_p11': results['p11']['true'],
    'Est_p11':  results['p11']['est'],
})

# Print the full trial-by-trial breakdown
print("\n--- Full Parameter Breakdown ---")
print(df_params.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# ------------------------------------------------------------
# Calculate and print the Average Error (Bias) across all trials
# ------------------------------------------------------------
print("\n--- Average Estimation Error (Est - True) ---")
error_summary = []
for key, title in metrics + [('p00', 'Persistence (Calm)'), ('p11', 'Persistence (Turbulent)')]:
    true_vals = np.array(results[key]['true'])
    est_vals = np.array(results[key]['est'])
    
    valid_idx = ~np.isnan(est_vals)
    mean_error = np.mean(est_vals[valid_idx] - true_vals[valid_idx])
    mae = np.mean(np.abs(est_vals[valid_idx] - true_vals[valid_idx]))
    
    error_summary.append({
        'Parameter': title,
        'Avg Bias': mean_error,
        'Mean Absolute Error': mae
    })

df_errors = pd.DataFrame(error_summary)
print(df_errors.to_string(index=False, float_format=lambda x: f"{x:.5f}"))